# Introduction

This notebook includes the solution for Fashion-MNIST using PyTorch.  

We are loading the data from `torchvision.datasets` (`Fashion-MNIST`).  

The model includes Batch Normalization and Dropout.



# Import packages

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Configuration & Data Loading

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

We will use a train/valid split (we divide the train data in train & validation set).


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Split training into train + validation
train_size = 55000
val_size = 5000
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 264kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.98MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.4MB/s]


# Model Definition

In [4]:
class FashionCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Conv Block 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Conv Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2, 2)

        # Dropout for conv features
        self.dropout_conv = nn.Dropout2d(0.25)

        # Fully connected
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout_fc = nn.Dropout(0.5)

        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        # Conv Block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Conv Block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        x = self.dropout_conv(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)

        x = self.fc2(x)

        return x


model = FashionCNN().to(device)

# Loss & Optimization

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Evaluation helper

We introduce an evaluation helper function so that we can evaluate at the end of each epoch the validation error by using the valid data.

In [6]:
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


# Train the model

In [7]:
epochs = 10
print_every = 200  # show training progress every 200 batches

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for step, (images, labels) in enumerate(train_loader, start=1):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accumulate stats
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

        # optional step-level reporting
        if step % print_every == 0:
            step_loss = running_loss / running_total
            step_acc = running_correct / running_total
            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Step [{step}/{len(train_loader)}] "
                f"Train Loss: {step_loss:.4f} "
                f"Train Acc: {step_acc:.4f}"
            )

    # end-of-epoch metrics
    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch [{epoch+1}/{epochs}] completed | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )

Epoch [1/10] Step [200/860] Train Loss: 0.7251 Train Acc: 0.7430
Epoch [1/10] Step [400/860] Train Loss: 0.6223 Train Acc: 0.7769
Epoch [1/10] Step [600/860] Train Loss: 0.5726 Train Acc: 0.7934
Epoch [1/10] Step [800/860] Train Loss: 0.5372 Train Acc: 0.8069
Epoch [1/10] completed | Train Loss: 0.5297, Train Acc: 0.8097 | Val Loss: 0.3277, Val Acc: 0.8788
Epoch [2/10] Step [200/860] Train Loss: 0.4001 Train Acc: 0.8546
Epoch [2/10] Step [400/860] Train Loss: 0.3988 Train Acc: 0.8552
Epoch [2/10] Step [600/860] Train Loss: 0.3906 Train Acc: 0.8574
Epoch [2/10] Step [800/860] Train Loss: 0.3826 Train Acc: 0.8601
Epoch [2/10] completed | Train Loss: 0.3813, Train Acc: 0.8608 | Val Loss: 0.2785, Val Acc: 0.8986
Epoch [3/10] Step [200/860] Train Loss: 0.3434 Train Acc: 0.8755
Epoch [3/10] Step [400/860] Train Loss: 0.3422 Train Acc: 0.8752
Epoch [3/10] Step [600/860] Train Loss: 0.3382 Train Acc: 0.8766
Epoch [3/10] Step [800/860] Train Loss: 0.3367 Train Acc: 0.8775
Epoch [3/10] completed

# Model Evaluation

In [8]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy (percent): {accuracy:.2f}%")

Test Accuracy (percent): 91.44%


# Final Remarks

We obtained a solution with an accuracy of 92% after 10 iterations.
The best validation score was close, of 92.8% (9th iteration).

Here are few ideas about how to improve the current solution:  
* Add more epochs to the training
* Add checkpointing and early stoping
* Add data augmentation to the pipeline

It will be great to also add Confusion Matrix and classification report to this Notebook, so that we can evaluate how the model is performing over all the 10 classes.
